# Neural Identifier Training with Particle Filter


## Library Imports

### This cell imports the essential libraries for the RHONN simulation:
 - **NumPy**: For numerical computations and array operations
 - **Plotly**: For interactive plotting and visualization of results


In [1]:

import numpy as np
import plotly.graph_objects as go

# True Nonlinear System (2-DOF Planar Robotic Arm Dynamics)

# This section defines the true nonlinear 2-DOF planar robotic arm system that we want to identify:
# - **`plant_dynamics()`**: Continuous-time 2-DOF arm dynamics using the Lagrangian formulation
# - **`plant()`**: Discrete-time implementation using Euler integration with configurable process noise (Laplacian, uniform, or Gaussian)

# The 2-DOF arm model includes:
# - State variables: theta1, omega1, theta2, omega2
# - Physical parameters: link masses (m1, m2), link lengths (l1, l2), damping (b1, b2), gravity (g)
# - Process noise for realistic simulation conditions
# ============================================================
# 1) True nonlinear system (2-DOF Planar Robotic Arm)
# ============================================================
def plant_dynamics(x, u, m1=1.0, m2=1.0, l1=1.0, l2=1.0, b1=0.1, b2=0.1, g=9.81):
    """
    Continuous dynamics for a 2-DOF planar arm:
    x = [theta1, omega1, theta2, omega2] (angles in rad, angular velocities in rad/s)
    u = [u1, u2] (Nm)
    Returns x_dot = [dtheta1/dt, domega1/dt, dtheta2/dt, domega2/dt].
    """
    theta1, omega1, theta2, omega2 = x

    # Inertia Matrix M(q)
    # M11 = (m1 + m2) * l1^2 + m2 * l2^2 + 2 * m2 * l1 * l2 * cos(theta2)
    # M12 = M21 = m2 * l2^2 + m2 * l1 * l2 * cos(theta2)
    # M22 = m2 * l2^2
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * np.cos(theta2)
    M12 = m2 * l2**2 + m2 * l1 * l2 * np.cos(theta2)
    M22 = m2 * l2**2
    det_M = M11 * M22 - M12**2
    if abs(det_M) < 1e-10:
        # Avoid singularity, though it's unlikely for these params
        det_M = 1e-10

    # Coriolis and Centrifugal Matrix C(q, q_dot)
    # C11 = -m2 * l1 * l2 * sin(theta2) * omega2
    # C12 = -m2 * l1 * l2 * sin(theta2) * (omega1 + omega2)
    # C21 = m2 * l1 * l2 * sin(theta2) * omega1
    # C22 = 0
    C11 = -m2 * l1 * l2 * np.sin(theta2) * omega2
    C12 = -m2 * l1 * l2 * np.sin(theta2) * (omega1 + omega2)
    C21 = m2 * l1 * l2 * np.sin(theta2) * omega1
    C22 = 0.0

    # Gravity Vector G(q)
    # G1 = (m1 + m2) * g * l1 * cos(theta1) + m2 * g * l2 * cos(theta1 + theta2)
    # G2 = m2 * g * l2 * cos(theta1 + theta2)
    G1 = (m1 + m2) * g * l1 * np.cos(theta1) + m2 * g * l2 * np.cos(theta1 + theta2)
    G2 = m2 * g * l2 * np.cos(theta1 + theta2)

    # Calculate q_ddot = M^(-1) * (u - C*q_dot - G)
    # M * [omega1_dot, omega2_dot]^T = u - C * [omega1, omega2]^T - G
    # We solve this system directly.
    # [M11, M12] [omega1_dot] = [u1 - C11*omega1 - C12*omega2 - G1]
    # [M21, M22] [omega2_dot] = [u2 - C21*omega1 - C22*omega2 - G2]
    # Using Cramer's rule or direct inversion:
    # omega1_dot = ( (u1 - C11*omega1 - C12*omega2 - G1) * M22 - (u2 - C21*omega1 - C22*omega2 - G2) * M12 ) / det_M
    # omega2_dot = ( (u2 - C21*omega1 - C22*omega2 - G2) * M11 - (u1 - C11*omega1 - C12*omega2 - G1) * M21 ) / det_M
    
    # Calculate terms
    tau1 = u[0] - C11 * omega1 - C12 * omega2 - G1
    tau2 = u[1] - C21 * omega1 - C22 * omega2 - G2
    
    omega1_dot = (tau1 * M22 - tau2 * M12) / det_M
    omega2_dot = (tau2 * M11 - tau1 * M12) / det_M # Note: M21 = M12

    # State derivatives
    theta1_dot = omega1
    theta2_dot = omega2
    
    return np.array([theta1_dot, omega1_dot, theta2_dot, omega2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise to all states
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# RHONN Structure and Feature Engineering

# This section implements the Recurrent High-Order Neural Network (RHONN) architecture:
# - **`sigmoidal()`**: Sigmoid activation function with numerical stability
# - **`construct_z_vector()`**: Creates the feature vector with high-order terms:
#   - Linear terms: S(x₁), S(x₂), S(x₃), S(x₄)
#   - Cross-products: S(x₁)S(x₂), S(x₁)S(x₃), S(x₁)S(x₄), S(x₂)S(x₃), S(x₂)S(x₄), S(x₃)S(x₄)
#   - Quadratic terms: S(x₁)², S(x₂)², S(x₃)², S(x₄)²
#   - Bias term: 1
# - **`RHONN_predict()`**: Forward pass for state prediction using neural network weights

# The RHONN uses series-parallel configuration for system identification.
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 4-state system (no inputs):
    z = [S(x1), S(x2), S(x3), S(x4), 
         S(x1)S(x2), S(x1)S(x3), S(x1)S(x4), S(x2)S(x3), S(x2)S(x4), S(x3)S(x4),
         S(x1)^2, S(x2)^2, S(x3)^2, S(x4)^2,
         1]
    Total: 15 features
    """
    s_x = [sigmoidal(x_est[i]) for i in range(4)]
    # Linear terms
    z_linear = s_x
    # Cross-product terms
    z_cross = [
        s_x[0] * s_x[1], s_x[0] * s_x[2], s_x[0] * s_x[3],
        s_x[1] * s_x[2], s_x[1] * s_x[3],
        s_x[2] * s_x[3]
    ]
    # Quadratic terms
    z_quad = [s**2 for s in s_x]
    # Bias term
    z_bias = [1.0]
    
    z = z_linear + z_cross + z_quad + z_bias
    return np.array(z)

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# Extended Kalman Filter (EKF) Trainer

# This class implements the Extended Kalman Filter for RHONN weight estimation:
# - **Weight dynamics**: Random walk model for neural network weights
# - **Series-parallel architecture**: Uses measured output states at time k for feature construction
# - **Kalman filtering**: Optimal linear estimation with Gaussian assumptions
# - **Covariance management**: Includes process noise (Q), measurement noise (R), and state covariance (P)
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        # Assume states 0 and 2 (theta1, theta2) are measured
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] # theta1 measured
        x_state_for_z[2] = chi_k[2] # theta2 measured
        # omega1 and omega2 are estimated

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

# Basic Particle Filter (PF) Trainer

# This class implements a standard Particle Filter for RHONN weight estimation:
# - **Predict**: Random walk evolution of particle weights
# - **Update**: Importance weight calculation using Gaussian likelihood
# - **Resample**: Systematic resampling when Effective Sample Size (ESS) drops
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        # Assume states 0 and 2 (theta1, theta2) are measured
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] # theta1 measured
        x_state_for_z[2] = chi_k[2] # theta2 measured
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

# Simulation Setup and Main Loop

# This section configures and executes the comparative simulation between EKF and PF approaches:
# - **Time horizon**: 2000 steps with dt=0.01s (20 seconds total)
# - **Initial conditions**: 30°, 0° for joint angles, zero angular velocities
# - **Process noise**: Laplacian distribution with configurable standard deviation
# - **RHONN configuration**: 4 neurons (one for each state), 15 features per neuron

# **Fair Comparison Setup:**
# - **Common initial weights**: Both methods start with identical weight initialization
# - **Series-parallel architecture**: Both use measured outputs (theta1, theta2) for feature construction
# - **Identical system dynamics**: Same true 2-DOF arm model for both approaches

# **Main Loop:**
# 1. Evolve true system dynamics
# 2. Update EKF weights and predict next state
# 3. Update PF weights and predict next state
# 4. Progress monitoring every 10% completion
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 2000 # Increase steps for longer simulation
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.005 # Reduce noise a bit for smoother simulation

    # --- True system init (2-DOF Arm) ---
    x_true = np.zeros((n_steps, 4)) # 4 states: theta1, omega1, theta2, omega2
    # Initial conditions: 30 degrees, 0 degrees, zero initial velocities
    x_true[0] = [np.pi / 6, 0.0, 0.0, 0.0]
    
    # Input torque signals - simple sinusoidal torques
    def input_torque1(t):
        # 0.5 Nm amplitude, 0.5 Hz frequency
        return 0.5 * np.sin(2 * np.pi * 0.5 * t)
    
    def input_torque2(t):
        # 0.3 Nm amplitude, 0.7 Hz frequency, phase shifted
        return 0.3 * np.sin(2 * np.pi * 0.7 * t + np.pi/4)

    # --- RHONN config ---
    num_neurons = 4 # One for each state (theta1_dot, omega1_dot, theta2_dot, omega2_dot)
    num_features = 15 # Based on construct_z_vector for 4 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i} (for state {i}): {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.5 # Tune R and eta
    )
    x_hat_ekf = np.zeros((n_steps, 4))
    x_hat_ekf[0] = x_true[0]

    # --- PF ---
    n_particles = 400
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.05, R_std=np.sqrt(5e-3), ess_threshold=n_particles / 2  # Tune R_std, match EKF R
    )

    # Force identical particle initialization
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation for 2-DOF Robotic Arm...")
    u1_history = [] # Store input torques for plotting
    u2_history = []
    for k in range(n_steps - 1):
        t_k = t_history[k]
        # ---- 1) Get input torques ----
        u1_k = input_torque1(t_k)
        u2_k = input_torque2(t_k)
        u_k = np.array([u1_k, u2_k])
        u1_history.append(u1_k)
        u2_history.append(u2_k)

        # ---- 2) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std)

        # ---- 3) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        # Prediction for next step (k+1) using updated weights
        # Series-parallel: use measured outputs (theta1, theta2) at k for z
        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0] # theta1 measured
        x_state_for_z_ekf[2] = x_true[k][2] # theta2 measured
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2])
        x_hat_ekf[k+1, 3] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[3])

        # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        # Prediction for next step (k+1) using updated weights
        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0] # theta1 measured
        x_state_for_z_pf[2] = x_true[k][2] # theta2 measured
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2])
        x_hat_pf[k+1, 3] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[3])

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

    # Append the last inputs
    u1_history.append(u1_history[-1])
    u2_history.append(u2_history[-1])

# Results Analysis and Visualization

# This section processes the simulation results and creates comprehensive visualizations:
# - **Mean Squared Error (MSE)**: Primary performance indicator for all state variables
# - **Final weight comparison**: Shows convergence of learned parameters
# - **State-by-state analysis**: Separate evaluation for each joint angle and velocity

# **Visualization Components:**
# 1. **State tracking plots**: True vs estimated trajectories for both methods
#    - Joint angle 1 tracking performance
#    - Joint angular velocity 1 tracking performance
#    - Joint angle 2 tracking performance
#    - Joint angular velocity 2 tracking performance
# 2. **Error analysis plot**: Time-series of identification errors
#    - Comparative error evolution for all states
#    - MSE values displayed in legend
# 3. **Input Torque Plots**: Visualization of applied control inputs

# **Plot Features:**
# - Interactive Plotly visualizations
# - Professional styling with clear legends
# - Comparative display of EKF vs PF performance
# - Time-domain analysis for full simulation duration
    # ============================================================
    # 6) Results & plots
    # ============================================================
    # Calculate MSE for each state
    mse_states_ekf = np.mean((x_true - x_hat_ekf)**2, axis=0)
    mse_states_pf = np.mean((x_true - x_hat_pf)**2, axis=0)

    print(f"\nFinal EKF-RHONN Weights:")
    for i in range(num_neurons):
        print(f"  Neuron {i} (for state {i}): {ekf_trainer.weights[i]}")
    print(f"\nFinal PF-RHONN Weight Estimates:")
    pf_final_weights = pf_trainer.get_estimate()
    for i in range(num_neurons):
        print(f"  Neuron {i} (for state {i}): {pf_final_weights[i]}")

    print("\n--- Performance Comparison (MSE) ---")
    state_names = ['Theta1 (Angle 1)', 'Omega1 (Vel. 1)', 'Theta2 (Angle 2)', 'Omega2 (Vel. 2)']
    for i in range(4):
        print(f"EKF MSE {state_names[i]}: {mse_states_ekf[i]:.6f}")
        print(f"PF  MSE {state_names[i]}: {mse_states_pf[i]:.6f}")

    # Plot States
    states_info = [
        {'idx': 0, 'var': 'θ₁', 'desc': 'Joint 1 Angle', 'y_label': 'Angle (rad)'},
        {'idx': 1, 'var': 'ω₁', 'desc': 'Joint 1 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
        {'idx': 2, 'var': 'θ₂', 'desc': 'Joint 2 Angle', 'y_label': 'Angle (rad)'},
        {'idx': 3, 'var': 'ω₂', 'desc': 'Joint 2 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
    ]

    for state_info in states_info:
        i = state_info['idx']
        trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                                 name=f'χ{i+1} (True {state_info["var"]})', line=dict(color='black', width=2))
        trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                              name=f'x{i+1} (Est. {state_info["var"]}, PF)', line=dict(dash='dot'))
        trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                               name=f'x{i+1} (Est. {state_info["var"]}, EKF)', line=dict(dash='dash'))

        fig = go.Figure([trace_plant, trace_pf, trace_ekf])
        fig.update_layout(
            title=f'2-DOF Arm Identification: {state_info["desc"]}',
            xaxis_title='Time (s)',
            yaxis_title=state_info['y_label'],
            legend=dict(x=0, y=1, orientation='h'),
            font=dict(size=12),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )
        fig.show()

    # Plot Errors for all states
    error_fig = go.Figure()
    for i in range(4):
        state_name = state_names[i]
        error_ekf = x_true[:, i] - x_hat_ekf[:, i]
        error_pf = x_true[:, i] - x_hat_pf[:, i]
        error_fig.add_trace(go.Scatter(x=t_history, y=error_ekf, mode='lines',
                              name=f'EKF Error {state_name} (MSE={mse_states_ekf[i]:.6f})', opacity=0.7))
        error_fig.add_trace(go.Scatter(x=t_history, y=error_pf, mode='lines',
                              name=f'PF Error {state_name} (MSE={mse_states_pf[i]:.6f})', opacity=0.7))
        
    error_fig.update_layout(
        title='Identification Errors for all 2-DOF Arm States',
        xaxis_title='Time (s)',
        yaxis_title='Error',
        legend=dict(x=0, y=1, orientation='v'), # Vertical legend for more states
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    error_fig.show()

    # Plot Input Torques
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=t_history, y=u1_history, mode='lines', name='Input Torque u1(t) (Nm)'))
    fig3.add_trace(go.Scatter(x=t_history, y=u2_history, mode='lines', name='Input Torque u2(t) (Nm)'))
    fig3.update_layout(
        title='Input Torques Applied to 2-DOF Robotic Arm',
        xaxis_title='Time (s)',
        yaxis_title='Torque (Nm)',
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig3.show()


Common Initial Weights:
  Neuron 0 (for state 0): [-0.21116517 -0.41193498  0.35740902  0.07630486  0.21621871 -0.04245175
 -0.2400971   0.07594791 -0.05948699  0.04680459 -0.03237007  0.09380008
 -0.45355818  0.33506331 -0.03182551]
  Neuron 1 (for state 1): [ 0.20473349  0.17984439  0.18925511 -0.38500731 -0.24497566  0.23278671
  0.40914724 -0.12824152  0.07128546 -0.06368446 -0.01492754  0.37298543
 -0.26280881  0.32572762 -0.32046131]
  Neuron 2 (for state 2): [-0.07315964  0.29408361 -0.27412484 -0.40753782  0.25550738  0.39674957
 -0.46167848  0.15917006 -0.27871822 -0.14998679  0.1630215   0.03150659
  0.26540083 -0.33411522 -0.28916292]
  Neuron 3 (for state 3): [ 0.11063965 -0.36077139  0.33103621 -0.21972467  0.22168466 -0.05374894
 -0.49317549  0.01813379  0.01313615 -0.46577296 -0.46988593 -0.18395309
  0.17916341  0.40742865 -0.01070765]
Starting simulation for 2-DOF Robotic Arm...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation 

## True Nonlinear System (2-DOF Planar Robotic Arm Dynamics)

This section defines the true nonlinear 2-DOF planar robotic arm system that we want to identify:
- **`plant_dynamics()`**: Continuous-time 2-DOF arm dynamics using the Lagrangian formulation
- **`plant()`**: Discrete-time implementation using Euler integration with configurable process noise

The 2-DOF arm model includes:
- State variables: θ₁, ω₁, θ₂, ω₂ (joint angles and angular velocities)
- Physical parameters: link masses (m₁, m₂), link lengths (l₁, l₂), damping (b₁, b₂), gravity (g)
- Control inputs: joint torques (u₁, u₂)
- Process noise for realistic simulation conditions

In [2]:
# ============================================================
# 1) True nonlinear system (2-DOF Planar Robotic Arm)
# ============================================================
def plant_dynamics(x, u, m1=1.0, m2=1.0, l1=1.0, l2=1.0, b1=0.1, b2=0.1, g=9.81):
    """
    Continuous dynamics for a 2-DOF planar arm:
    x = [theta1, omega1, theta2, omega2] (angles in rad, angular velocities in rad/s)
    u = [u1, u2] (Nm)
    Returns x_dot = [dtheta1/dt, domega1/dt, dtheta2/dt, domega2/dt].
    """
    theta1, omega1, theta2, omega2 = x

    # Inertia Matrix M(q)
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * np.cos(theta2)
    M12 = m2 * l2**2 + m2 * l1 * l2 * np.cos(theta2)
    M22 = m2 * l2**2
    det_M = M11 * M22 - M12**2
    if abs(det_M) < 1e-10:
        # Avoid singularity, though it's unlikely for these params
        det_M = 1e-10

    # Coriolis and Centrifugal Matrix C(q, q_dot)
    C11 = -m2 * l1 * l2 * np.sin(theta2) * omega2
    C12 = -m2 * l1 * l2 * np.sin(theta2) * (omega1 + omega2)
    C21 = m2 * l1 * l2 * np.sin(theta2) * omega1
    C22 = 0.0

    # Gravity Vector G(q)
    G1 = (m1 + m2) * g * l1 * np.cos(theta1) + m2 * g * l2 * np.cos(theta1 + theta2)
    G2 = m2 * g * l2 * np.cos(theta1 + theta2)

    # Calculate q_ddot = M^(-1) * (u - C*q_dot - G)
    tau1 = u[0] - C11 * omega1 - C12 * omega2 - G1
    tau2 = u[1] - C21 * omega1 - C22 * omega2 - G2
    
    omega1_dot = (tau1 * M22 - tau2 * M12) / det_M
    omega2_dot = (tau2 * M11 - tau1 * M12) / det_M # Note: M21 = M12

    # State derivatives
    theta1_dot = omega1
    theta2_dot = omega2
    
    return np.array([theta1_dot, omega1_dot, theta2_dot, omega2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise to all states
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

## RHONN Structure and Neural Network Functions

In [3]:
# ============================================================
# 2) RHONN structure for 2-DOF robot (4 states + 2 control inputs)
# ============================================================

def construct_z_vector(x_k, u_k, max_lags=2):
    """
    Construct high-order polynomial features for RHONN.
    For a 2-DOF robotic arm: x_k = [theta1, omega1, theta2, omega2], u_k = [u1, u2]
    """
    theta1, omega1, theta2, omega2 = x_k
    u1, u2 = u_k
    
    z = [
        # Linear terms (state + control)
        theta1, omega1, theta2, omega2, u1, u2,
        
        # Quadratic state terms
        theta1**2, omega1**2, theta2**2, omega2**2,
        
        # Cross terms between states
        theta1*omega1, theta1*theta2, theta1*omega2,
        omega1*theta2, omega1*omega2, theta2*omega2,
        
        # Control interaction terms
        u1*u2
    ]
    
    return np.array(z)

def RHONN_predict(w, x_k, u_k):
    """
    RHONN forward pass using polynomial features.
    Returns x_{k+1} prediction.
    """
    z_k = construct_z_vector(x_k, u_k)
    n_states = 4  # 2-DOF robot has 4 states
    
    x_kp1_pred = np.zeros(n_states)
    feature_len = len(z_k)
    
    for i in range(n_states):
        w_i = w[i * feature_len:(i + 1) * feature_len]
        x_kp1_pred[i] = np.dot(w_i, z_k)
    
    return x_kp1_pred

## Extended Kalman Filter (EKF) Trainer

In [4]:
# ============================================================
# 3) EKF-based RHONN trainer for robotic system
# ============================================================

class EKF_RHONN_Trainer:
    def __init__(self, n_features=15, n_states=4, Q_scale=1e-5, R_scale=1e-3):
        """
        EKF trainer for 2-DOF robotic arm RHONN.
        n_features: Number of polynomial features (15 for our case)
        n_states: Number of states (4 for 2-DOF robot)
        """
        self.n_features = n_features
        self.n_states = n_states
        self.n_weights = n_features * n_states
        
        # Initialize weights
        self.w = np.random.normal(0, 0.1, self.n_weights)
        
        # Covariance matrix
        self.P = np.eye(self.n_weights) * 1.0
        
        # Process and measurement noise
        self.Q = np.eye(self.n_weights) * Q_scale
        self.R = np.eye(n_states) * R_scale
        
        # Storage
        self.mse_history = []
    
    def compute_jacobian(self, x_k, u_k):
        """Compute Jacobian of RHONN output w.r.t. weights."""
        z_k = construct_z_vector(x_k, u_k)
        
        H = np.zeros((self.n_states, self.n_weights))
        for i in range(self.n_states):
            start_idx = i * self.n_features
            end_idx = (i + 1) * self.n_features
            H[i, start_idx:end_idx] = z_k
        
        return H
    
    def predict(self):
        """EKF prediction step."""
        # For weights, F = I (random walk model)
        # self.w = self.w (no change)
        self.P = self.P + self.Q
    
    def update(self, x_k, u_k, x_target):
        """EKF update step."""
        # Prediction
        x_pred = RHONN_predict(self.w, x_k, u_k)
        
        # Innovation
        y = x_target - x_pred
        
        # Jacobian
        H = self.compute_jacobian(x_k, u_k)
        
        # Innovation covariance
        S = H @ self.P @ H.T + self.R
        
        # Kalman gain
        try:
            K = self.P @ H.T @ np.linalg.inv(S)
        except np.linalg.LinAlgError:
            K = self.P @ H.T @ np.linalg.pinv(S)
        
        # Update weights and covariance
        self.w = self.w + K @ y
        self.P = (np.eye(self.n_weights) - K @ H) @ self.P
        
        # Store MSE
        mse = np.mean(y**2)
        self.mse_history.append(mse)
        
        return mse

## Enhanced Particle Filter (PF) Trainer

In [ ]:
# ============================================================
# 4) Enhanced Particle Filter (PF) RHONN trainer for robotic system
# ============================================================

class EnhancedPF_RHONN_Trainer:
    def __init__(self, num_neurons=4, num_weights_per_neuron=15, n_particles=500, 
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        """
        Enhanced Particle Filter trainer for 2-DOF robotic arm RHONN with multiple improvements.
        """
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        
        # Dynamic particle count
        self.n_particles = n_particles
        self.min_particles = max(100, n_particles // 5)
        self.max_particles = n_particles * 2
        
        # ESS threshold for resampling
        if ess_threshold is None:
            ess_threshold = n_particles // 3
        self.ess_threshold = ess_threshold
        
        # Initialize particles (weights for each neuron)
        self.particles = []
        self.weights_pf = []
        
        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)
        
        # Noise parameters (adaptive)
        self.base_Q_std = Q_std
        self.base_R_std = R_std
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        
        # Storage
        self.mse_history = []
        self.ess_history = []
        self.particle_count_history = []
        
        # Adaptive parameters
        self.performance_window = 10
        self.last_mse_values = []
        
        # Pre-training phase
        self.pre_training_steps = 50
        self.step_count = 0
    
    def _ess(self, w):
        """Compute effective sample size."""
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)
    
    def _resample_systematic(self, neuron_index):
        """Systematic resampling for a specific neuron."""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N
    
    def adaptive_noise_scaling(self):
        """Dynamically adjust noise based on recent performance."""
        if len(self.last_mse_values) < 3:
            return 1.0
        
        recent_trend = np.mean(np.diff(self.last_mse_values[-5:]))
        
        if recent_trend > 0:  # Performance degrading
            scale = 1.2
        elif recent_trend < -0.001:  # Performance improving rapidly
            scale = 0.9
        else:  # Stable
            scale = 1.0
        
        return np.clip(scale, 0.5, 2.0)
    
    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        Enhanced PF update step for all neurons.
        
        chi_kp1: measured true states at k+1 (targets)
        chi_k: measured true states at k (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        self.step_count += 1
        
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta1 measured
        x_state_for_z[2] = chi_k[2]  # theta2 measured
        z = construct_z_vector(x_state_for_z)
        
        # Adaptive noise scaling
        noise_scale = self.adaptive_noise_scaling()
        current_Q_std = self.base_Q_std * noise_scale
        
        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * current_Q_std
        
        # 2) Update: importance weights with Gaussian likelihood
        total_mse = 0
        for i in range(self.num_neurons):
            w_mat = self.particles[i]  # (N, num_features)
            x_pred_particles = w_mat @ z  # (N,)
            innov = chi_kp1[i] - x_pred_particles
            
            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)
            
            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s
            
            # 3) Resample if ESS is low
            ess = self._ess(self.weights_pf[i])
            if ess < self.ess_threshold:
                self._resample_systematic(i)
            
            # Track MSE for this neuron
            pred_mean = np.mean(x_pred_particles)
            total_mse += (chi_kp1[i] - pred_mean)**2
        
        # Store metrics
        mse = total_mse / self.num_neurons
        self.mse_history.append(mse)
        self.last_mse_values.append(mse)
        
        # Keep window size manageable
        if len(self.last_mse_values) > self.performance_window:
            self.last_mse_values.pop(0)
        
        # Store ESS and particle count
        avg_ess = np.mean([self._ess(self.weights_pf[i]) for i in range(self.num_neurons)])
        self.ess_history.append(avg_ess)
        self.particle_count_history.append(self.n_particles)
        
        return mse
    
    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

## Simulation Setup and Reference Trajectory

In [ ]:
# ============================================================
# 5) Simulation Setup and Execution
# ============================================================

# --- Simulation settings ---
n_steps = 2000  # 20 seconds total with dt=0.01
dt = 0.01
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.005  # Reduced noise for smoother simulation

# --- True system initialization (2-DOF Arm) ---
x_true = np.zeros((n_steps, 4))  # 4 states: theta1, omega1, theta2, omega2
# Initial conditions: 30 degrees for joint 1, 0 degrees for joint 2, zero velocities
x_true[0] = [np.pi / 6, 0.0, 0.0, 0.0]

# Input torque signals - sinusoidal reference tracking
def input_torque1(t):
    return 0.5 * np.sin(2 * np.pi * 0.5 * t)  # 0.5 Nm amplitude, 0.5 Hz frequency

def input_torque2(t):
    return 0.3 * np.sin(2 * np.pi * 0.7 * t + np.pi/4)  # 0.3 Nm amplitude, 0.7 Hz, phase shifted

# --- RHONN configuration ---
num_neurons = 4  # One for each state
num_features = 15  # Based on construct_z_vector for 4 states
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i} (for state {i}): {w}")

# --- Initialize EKF Trainer ---
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.5
)
x_hat_ekf = np.zeros((n_steps, 4))
x_hat_ekf[0] = x_true[0]

# --- Initialize Enhanced PF Trainer ---
n_particles = 400
pf_trainer = EnhancedPF_RHONN_Trainer(
    num_neurons=num_neurons, 
    num_weights_per_neuron=num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=0.05, R_std=np.sqrt(5e-3),
    ess_threshold=n_particles // 3
)

x_hat_pf = np.zeros((n_steps, 4))
x_hat_pf[0] = x_true[0]

print("Starting simulation for 2-DOF Robotic Arm...")

# Storage for control inputs
u1_history = []
u2_history = []

# --- Main simulation loop ---
for k in range(n_steps - 1):
    t_k = t_history[k]
    
    # Get input torques
    u1_k = input_torque1(t_k)
    u2_k = input_torque2(t_k)
    u_k = np.array([u1_k, u2_k])
    u1_history.append(u1_k)
    u2_history.append(u2_k)

    # Evolve true system
    x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std)

    # EKF update and prediction
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])
    
    # EKF prediction using series-parallel architecture
    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_true[k][0]  # theta1 measured
    x_state_for_z_ekf[2] = x_true[k][2]  # theta2 measured
    
    for i in range(num_neurons):
        x_hat_ekf[k+1, i] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[i])

    # Enhanced PF update and prediction
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])
    
    # PF prediction using weight estimates
    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_true[k][0]  # theta1 measured
    x_state_for_z_pf[2] = x_true[k][2]  # theta2 measured
    
    for i in range(num_neurons):
        x_hat_pf[k+1, i] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[i])

    # Progress monitoring
    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

# Append final control inputs
u1_history.append(u1_history[-1])
u2_history.append(u2_history[-1])

Common Initial Weights:
  Neuron 0 (for state 0): [ 0.11329932 -0.47641851 -0.05463042 -0.12214805  0.14827538 -0.39348222
 -0.19796093 -0.35213674  0.31294296 -0.34179651  0.34366128 -0.43383417
  0.30738079 -0.20606038 -0.08834454]
  Neuron 1 (for state 1): [ 0.17013254  0.42037672 -0.41770695  0.20915896 -0.23950847  0.46785525
 -0.4281463  -0.37605063 -0.20168954  0.24216708  0.20479104 -0.33062317
  0.23182193 -0.02959899 -0.349909  ]
  Neuron 2 (for state 2): [ 0.27196488  0.08656226  0.41161393 -0.31339374 -0.06724572 -0.17853432
 -0.13763727  0.36080693  0.10936491 -0.34631756  0.08590926  0.12461679
  0.44804647  0.19029095  0.22945456]
  Neuron 3 (for state 3): [ 0.40487188 -0.42303158 -0.10972152  0.14990515 -0.08577399 -0.11603249
  0.48669101 -0.01601771 -0.17435009 -0.15361821  0.11000026 -0.22268466
  0.08357481  0.05983952 -0.48824934]


TypeError: EKF_RHONN_Trainer.__init__() got an unexpected keyword argument 'initial_weights'

## Results Analysis and Visualization

In [ ]:
# ============================================================
# 6) Results Analysis and Comprehensive Visualization
# ============================================================

# Calculate MSE for each state
mse_states_ekf = np.mean((x_true - x_hat_ekf)**2, axis=0)
mse_states_pf = np.mean((x_true - x_hat_pf)**2, axis=0)

# Print final weight estimates
print(f"\nFinal EKF-RHONN Weights:")
for i in range(num_neurons):
    print(f"  Neuron {i} (for state {i}): {ekf_trainer.weights[i]}")

print(f"\nFinal Enhanced PF-RHONN Weight Estimates:")
pf_final_weights = pf_trainer.get_estimate()
for i in range(num_neurons):
    print(f"  Neuron {i} (for state {i}): {pf_final_weights[i]}")

# Performance comparison
print("\n--- Performance Comparison (MSE) ---")
state_names = ['Theta1 (Joint 1 Angle)', 'Omega1 (Joint 1 Velocity)', 
               'Theta2 (Joint 2 Angle)', 'Omega2 (Joint 2 Velocity)']

for i in range(4):
    improvement = ((mse_states_ekf[i] - mse_states_pf[i]) / mse_states_ekf[i] * 100)
    print(f"{state_names[i]}:")
    print(f"  EKF MSE: {mse_states_ekf[i]:.6f}")
    print(f"  Enhanced PF MSE: {mse_states_pf[i]:.6f}")
    print(f"  Improvement: {improvement:.2f}%")
    print()

# Enhanced PF performance metrics
print("--- Enhanced PF Performance Metrics ---")
print(f"Final ESS: {pf_trainer.ess_history[-1]:.2f}")
print(f"Average ESS: {np.mean(pf_trainer.ess_history):.2f}")
print(f"Final particle count: {pf_trainer.particle_count_history[-1]}")
if len(pf_trainer.mse_history) > 200:
    print(f"MSE reduction over last 100 steps: {(np.mean(pf_trainer.mse_history[-200:-100]) - np.mean(pf_trainer.mse_history[-100:])) / np.mean(pf_trainer.mse_history[-200:-100]) * 100:.2f}%")

# Plot individual state tracking
states_info = [
    {'idx': 0, 'var': 'θ₁', 'desc': 'Joint 1 Angle', 'y_label': 'Angle (rad)'},
    {'idx': 1, 'var': 'ω₁', 'desc': 'Joint 1 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
    {'idx': 2, 'var': 'θ₂', 'desc': 'Joint 2 Angle', 'y_label': 'Angle (rad)'},
    {'idx': 3, 'var': 'ω₂', 'desc': 'Joint 2 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
]

for state_info in states_info:
    i = state_info['idx']
    mse_ekf = mse_states_ekf[i]
    mse_pf = mse_states_pf[i]
    improvement = ((mse_ekf - mse_pf) / mse_ekf * 100)
    
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=f'True {state_info["var"]}', 
                            line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                          name=f'EKF Est. (MSE={mse_ekf:.6f})', 
                          line=dict(dash='dash', color='blue'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                         name=f'Enhanced PF Est. (MSE={mse_pf:.6f}, +{improvement:.1f}%)', 
                         line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_pf])
    fig.update_layout(
        title=f'2-DOF Robotic Arm Identification: {state_info["desc"]}',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Comprehensive error analysis plot
error_fig = go.Figure()
colors = ['blue', 'lightblue', 'red', 'lightcoral']

for i in range(4):
    state_name = state_names[i]
    error_ekf = x_true[:, i] - x_hat_ekf[:, i]
    error_pf = x_true[:, i] - x_hat_pf[:, i]
    
    error_fig.add_trace(go.Scatter(x=t_history, y=error_ekf, mode='lines',
                          name=f'EKF Error {state_name} (MSE={mse_states_ekf[i]:.6f})', 
                          line=dict(color=colors[i % 2], dash='dash'),
                          opacity=0.7))
    error_fig.add_trace(go.Scatter(x=t_history, y=error_pf, mode='lines',
                          name=f'Enhanced PF Error {state_name} (MSE={mse_states_pf[i]:.6f})', 
                          line=dict(color=colors[i % 2 + 2]),
                          opacity=0.7))
        
error_fig.update_layout(
    title='Identification Errors for All 2-DOF Arm States (EKF vs Enhanced PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='v'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
error_fig.show()

# Control input visualization
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=u1_history, mode='lines', 
                         name='Input Torque u₁(t) (Joint 1)', 
                         line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=u2_history, mode='lines', 
                         name='Input Torque u₂(t) (Joint 2)', 
                         line=dict(color='orange')))
fig2.update_layout(
    title='Control Inputs Applied to 2-DOF Robotic Arm',
    xaxis_title='Time (s)',
    yaxis_title='Torque (Nm)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# Enhanced PF-specific performance plots
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=list(range(len(pf_trainer.mse_history))), 
                         y=pf_trainer.mse_history, mode='lines',
                         name='Enhanced PF MSE Evolution'))
fig3.update_layout(
    title='Enhanced PF Training Progress (MSE over Time)',
    xaxis_title='Training Step',
    yaxis_title='MSE',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig3.show()

# Summary statistics
print("\n=== FINAL SUMMARY ===")
total_mse_ekf = np.mean(mse_states_ekf)
total_mse_pf = np.mean(mse_states_pf)
overall_improvement = ((total_mse_ekf - total_mse_pf) / total_mse_ekf * 100)

print(f"Overall EKF MSE (average): {total_mse_ekf:.6f}")
print(f"Overall Enhanced PF MSE (average): {total_mse_pf:.6f}")
print(f"Overall improvement: {overall_improvement:.2f}%")
print(f"Enhanced PF outperforms EKF: {'✓ YES' if total_mse_pf < total_mse_ekf else '✗ NO'}")

In [ ]:

# This cell imports the essential libraries for the RHONN simulation:
# - **NumPy**: For numerical computations and array operations
# - **Plotly**: For interactive plotting and visualization of results
import numpy as np
import plotly.graph_objects as go

# True Nonlinear System (2-DOF Planar Robotic Arm Dynamics)

# This section defines the true nonlinear 2-DOF planar robotic arm system that we want to identify:
# - **`plant_dynamics()`**: Continuous-time 2-DOF arm dynamics using the Lagrangian formulation
# - **`plant()`**: Discrete-time implementation using Euler integration with configurable process noise (Laplacian, uniform, or Gaussian)

# The 2-DOF arm model includes:
# - State variables: theta1, omega1, theta2, omega2
# - Physical parameters: link masses (m1, m2), link lengths (l1, l2), damping (b1, b2), gravity (g)
# - Process noise for realistic simulation conditions
# ============================================================
# 1) True nonlinear system (2-DOF Planar Robotic Arm)
# ============================================================
def plant_dynamics(x, u, m1=1.0, m2=1.0, l1=1.0, l2=1.0, b1=0.1, b2=0.1, g=9.81):
    """
    Continuous dynamics for a 2-DOF planar arm:
    x = [theta1, omega1, theta2, omega2] (angles in rad, angular velocities in rad/s)
    u = [u1, u2] (Nm)
    Returns x_dot = [dtheta1/dt, domega1/dt, dtheta2/dt, domega2/dt].
    """
    theta1, omega1, theta2, omega2 = x

    # Inertia Matrix M(q)
    # M11 = (m1 + m2) * l1^2 + m2 * l2^2 + 2 * m2 * l1 * l2 * cos(theta2)
    # M12 = M21 = m2 * l2^2 + m2 * l1 * l2 * cos(theta2)
    # M22 = m2 * l2^2
    M11 = (m1 + m2) * l1**2 + m2 * l2**2 + 2 * m2 * l1 * l2 * np.cos(theta2)
    M12 = m2 * l2**2 + m2 * l1 * l2 * np.cos(theta2)
    M22 = m2 * l2**2
    det_M = M11 * M22 - M12**2
    if abs(det_M) < 1e-10:
        # Avoid singularity, though it's unlikely for these params
        det_M = 1e-10

    # Coriolis and Centrifugal Matrix C(q, q_dot)
    # C11 = -m2 * l1 * l2 * sin(theta2) * omega2
    # C12 = -m2 * l1 * l2 * sin(theta2) * (omega1 + omega2)
    # C21 = m2 * l1 * l2 * sin(theta2) * omega1
    # C22 = 0
    C11 = -m2 * l1 * l2 * np.sin(theta2) * omega2
    C12 = -m2 * l1 * l2 * np.sin(theta2) * (omega1 + omega2)
    C21 = m2 * l1 * l2 * np.sin(theta2) * omega1
    C22 = 0.0

    # Gravity Vector G(q)
    # G1 = (m1 + m2) * g * l1 * cos(theta1) + m2 * g * l2 * cos(theta1 + theta2)
    # G2 = m2 * g * l2 * cos(theta1 + theta2)
    G1 = (m1 + m2) * g * l1 * np.cos(theta1) + m2 * g * l2 * np.cos(theta1 + theta2)
    G2 = m2 * g * l2 * np.cos(theta1 + theta2)

    # Calculate q_ddot = M^(-1) * (u - C*q_dot - G)
    # M * [omega1_dot, omega2_dot]^T = u - C * [omega1, omega2]^T - G
    # We solve this system directly.
    # [M11, M12] [omega1_dot] = [u1 - C11*omega1 - C12*omega2 - G1]
    # [M21, M22] [omega2_dot] = [u2 - C21*omega1 - C22*omega2 - G2]
    # Using Cramer's rule or direct inversion:
    # omega1_dot = ( (u1 - C11*omega1 - C12*omega2 - G1) * M22 - (u2 - C21*omega1 - C22*omega2 - G2) * M12 ) / det_M
    # omega2_dot = ( (u2 - C21*omega1 - C22*omega2 - G2) * M11 - (u1 - C11*omega1 - C12*omega2 - G1) * M21 ) / det_M
    
    # Calculate terms
    tau1 = u[0] - C11 * omega1 - C12 * omega2 - G1
    tau2 = u[1] - C21 * omega1 - C22 * omega2 - G2
    
    omega1_dot = (tau1 * M22 - tau2 * M12) / det_M
    omega2_dot = (tau2 * M11 - tau1 * M12) / det_M # Note: M21 = M12

    # State derivatives
    theta1_dot = omega1
    theta2_dot = omega2
    
    return np.array([theta1_dot, omega1_dot, theta2_dot, omega2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise to all states
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# RHONN Structure and Feature Engineering

# This section implements the Recurrent High-Order Neural Network (RHONN) architecture:
# - **`sigmoidal()`**: Sigmoid activation function with numerical stability
# - **`construct_z_vector()`**: Creates the feature vector with high-order terms:
#   - Linear terms: S(x₁), S(x₂), S(x₃), S(x₄)
#   - Cross-products: S(x₁)S(x₂), S(x₁)S(x₃), S(x₁)S(x₄), S(x₂)S(x₃), S(x₂)S(x₄), S(x₃)S(x₄)
#   - Quadratic terms: S(x₁)², S(x₂)², S(x₃)², S(x₄)²
#   - Bias term: 1
# - **`RHONN_predict()`**: Forward pass for state prediction using neural network weights

# The RHONN uses series-parallel configuration for system identification.
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 4-state system (no inputs):
    z = [S(x1), S(x2), S(x3), S(x4), 
         S(x1)S(x2), S(x1)S(x3), S(x1)S(x4), S(x2)S(x3), S(x2)S(x4), S(x3)S(x4),
         S(x1)^2, S(x2)^2, S(x3)^2, S(x4)^2,
         1]
    Total: 15 features
    """
    s_x = [sigmoidal(x_est[i]) for i in range(4)]
    # Linear terms
    z_linear = s_x
    # Cross-product terms
    z_cross = [
        s_x[0] * s_x[1], s_x[0] * s_x[2], s_x[0] * s_x[3],
        s_x[1] * s_x[2], s_x[1] * s_x[3],
        s_x[2] * s_x[3]
    ]
    # Quadratic terms
    z_quad = [s**2 for s in s_x]
    # Bias term
    z_bias = [1.0]
    
    z = z_linear + z_cross + z_quad + z_bias
    return np.array(z)

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# Extended Kalman Filter (EKF) Trainer

# This class implements the Extended Kalman Filter for RHONN weight estimation:
# - **Weight dynamics**: Random walk model for neural network weights
# - **Series-parallel architecture**: Uses measured output states at time k for feature construction
# - **Kalman filtering**: Optimal linear estimation with Gaussian assumptions
# - **Covariance management**: Includes process noise (Q), measurement noise (R), and state covariance (P)
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        # Assume states 0 and 2 (theta1, theta2) are measured
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] # theta1 measured
        x_state_for_z[2] = chi_k[2] # theta2 measured
        # omega1 and omega2 are estimated

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

# Basic Particle Filter (PF) Trainer

# This class implements a standard Particle Filter for RHONN weight estimation:
# - **Predict**: Random walk evolution of particle weights
# - **Update**: Importance weight calculation using Gaussian likelihood
# - **Resample**: Systematic resampling when Effective Sample Size (ESS) drops
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        # Assume states 0 and 2 (theta1, theta2) are measured
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] # theta1 measured
        x_state_for_z[2] = chi_k[2] # theta2 measured
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

# Simulation Setup and Main Loop

# This section configures and executes the comparative simulation between EKF and PF approaches:
# - **Time horizon**: 2000 steps with dt=0.01s (20 seconds total)
# - **Initial conditions**: 30°, 0° for joint angles, zero angular velocities
# - **Process noise**: Laplacian distribution with configurable standard deviation
# - **RHONN configuration**: 4 neurons (one for each state), 15 features per neuron

# **Fair Comparison Setup:**
# - **Common initial weights**: Both methods start with identical weight initialization
# - **Series-parallel architecture**: Both use measured outputs (theta1, theta2) for feature construction
# - **Identical system dynamics**: Same true 2-DOF arm model for both approaches

# **Main Loop:**
# 1. Evolve true system dynamics
# 2. Update EKF weights and predict next state
# 3. Update PF weights and predict next state
# 4. Progress monitoring every 10% completion
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 2000 # Increase steps for longer simulation
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.005 # Reduce noise a bit for smoother simulation

    # --- True system init (2-DOF Arm) ---
    x_true = np.zeros((n_steps, 4)) # 4 states: theta1, omega1, theta2, omega2
    # Initial conditions: 30 degrees, 0 degrees, zero initial velocities
    x_true[0] = [np.pi / 6, 0.0, 0.0, 0.0]
    
    # Input torque signals - simple sinusoidal torques
    def input_torque1(t):
        # 0.5 Nm amplitude, 0.5 Hz frequency
        return 0.5 * np.sin(2 * np.pi * 0.5 * t)
    
    def input_torque2(t):
        # 0.3 Nm amplitude, 0.7 Hz frequency, phase shifted
        return 0.3 * np.sin(2 * np.pi * 0.7 * t + np.pi/4)

    # --- RHONN config ---
    num_neurons = 4 # One for each state (theta1_dot, omega1_dot, theta2_dot, omega2_dot)
    num_features = 15 # Based on construct_z_vector for 4 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i} (for state {i}): {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=5e-3, P_init=1.0, eta=0.5 # Tune R and eta
    )
    x_hat_ekf = np.zeros((n_steps, 4))
    x_hat_ekf[0] = x_true[0]

    # --- PF ---
    n_particles = 400
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.05, R_std=np.sqrt(5e-3), ess_threshold=n_particles / 2  # Tune R_std, match EKF R
    )

    # Force identical particle initialization
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation for 2-DOF Robotic Arm...")
    u1_history = [] # Store input torques for plotting
    u2_history = []
    for k in range(n_steps - 1):
        t_k = t_history[k]
        # ---- 1) Get input torques ----
        u1_k = input_torque1(t_k)
        u2_k = input_torque2(t_k)
        u_k = np.array([u1_k, u2_k])
        u1_history.append(u1_k)
        u2_history.append(u2_k)

        # ---- 2) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std)

        # ---- 3) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        # Prediction for next step (k+1) using updated weights
        # Series-parallel: use measured outputs (theta1, theta2) at k for z
        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0] # theta1 measured
        x_state_for_z_ekf[2] = x_true[k][2] # theta2 measured
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2])
        x_hat_ekf[k+1, 3] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[3])

        # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        # Prediction for next step (k+1) using updated weights
        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0] # theta1 measured
        x_state_for_z_pf[2] = x_true[k][2] # theta2 measured
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2])
        x_hat_pf[k+1, 3] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[3])

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

    # Append the last inputs
    u1_history.append(u1_history[-1])
    u2_history.append(u2_history[-1])

# Results Analysis and Visualization

# This section processes the simulation results and creates comprehensive visualizations:
# - **Mean Squared Error (MSE)**: Primary performance indicator for all state variables
# - **Final weight comparison**: Shows convergence of learned parameters
# - **State-by-state analysis**: Separate evaluation for each joint angle and velocity

# **Visualization Components:**
# 1. **State tracking plots**: True vs estimated trajectories for both methods
#    - Joint angle 1 tracking performance
#    - Joint angular velocity 1 tracking performance
#    - Joint angle 2 tracking performance
#    - Joint angular velocity 2 tracking performance
# 2. **Error analysis plot**: Time-series of identification errors
#    - Comparative error evolution for all states
#    - MSE values displayed in legend
# 3. **Input Torque Plots**: Visualization of applied control inputs

# **Plot Features:**
# - Interactive Plotly visualizations
# - Professional styling with clear legends
# - Comparative display of EKF vs PF performance
# - Time-domain analysis for full simulation duration
    # ============================================================
    # 6) Results & plots
    # ============================================================
    # Calculate MSE for each state
    mse_states_ekf = np.mean((x_true - x_hat_ekf)**2, axis=0)
    mse_states_pf = np.mean((x_true - x_hat_pf)**2, axis=0)

    print(f"\nFinal EKF-RHONN Weights:")
    for i in range(num_neurons):
        print(f"  Neuron {i} (for state {i}): {ekf_trainer.weights[i]}")
    print(f"\nFinal PF-RHONN Weight Estimates:")
    pf_final_weights = pf_trainer.get_estimate()
    for i in range(num_neurons):
        print(f"  Neuron {i} (for state {i}): {pf_final_weights[i]}")

    print("\n--- Performance Comparison (MSE) ---")
    state_names = ['Theta1 (Angle 1)', 'Omega1 (Vel. 1)', 'Theta2 (Angle 2)', 'Omega2 (Vel. 2)']
    for i in range(4):
        print(f"EKF MSE {state_names[i]}: {mse_states_ekf[i]:.6f}")
        print(f"PF  MSE {state_names[i]}: {mse_states_pf[i]:.6f}")

    # Plot States
    states_info = [
        {'idx': 0, 'var': 'θ₁', 'desc': 'Joint 1 Angle', 'y_label': 'Angle (rad)'},
        {'idx': 1, 'var': 'ω₁', 'desc': 'Joint 1 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
        {'idx': 2, 'var': 'θ₂', 'desc': 'Joint 2 Angle', 'y_label': 'Angle (rad)'},
        {'idx': 3, 'var': 'ω₂', 'desc': 'Joint 2 Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)'},
    ]

    for state_info in states_info:
        i = state_info['idx']
        trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                                 name=f'χ{i+1} (True {state_info["var"]})', line=dict(color='black', width=2))
        trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                              name=f'x{i+1} (Est. {state_info["var"]}, PF)', line=dict(dash='dot'))
        trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                               name=f'x{i+1} (Est. {state_info["var"]}, EKF)', line=dict(dash='dash'))

        fig = go.Figure([trace_plant, trace_pf, trace_ekf])
        fig.update_layout(
            title=f'2-DOF Arm Identification: {state_info["desc"]}',
            xaxis_title='Time (s)',
            yaxis_title=state_info['y_label'],
            legend=dict(x=0, y=1, orientation='h'),
            font=dict(size=12),
            plot_bgcolor='white',
            paper_bgcolor='white'
        )
        fig.show()

    # Plot Errors for all states
    error_fig = go.Figure()
    for i in range(4):
        state_name = state_names[i]
        error_ekf = x_true[:, i] - x_hat_ekf[:, i]
        error_pf = x_true[:, i] - x_hat_pf[:, i]
        error_fig.add_trace(go.Scatter(x=t_history, y=error_ekf, mode='lines',
                              name=f'EKF Error {state_name} (MSE={mse_states_ekf[i]:.6f})', opacity=0.7))
        error_fig.add_trace(go.Scatter(x=t_history, y=error_pf, mode='lines',
                              name=f'PF Error {state_name} (MSE={mse_states_pf[i]:.6f})', opacity=0.7))
        
    error_fig.update_layout(
        title='Identification Errors for all 2-DOF Arm States',
        xaxis_title='Time (s)',
        yaxis_title='Error',
        legend=dict(x=0, y=1, orientation='v'), # Vertical legend for more states
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    error_fig.show()

    # Plot Input Torques
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=t_history, y=u1_history, mode='lines', name='Input Torque u1(t) (Nm)'))
    fig3.add_trace(go.Scatter(x=t_history, y=u2_history, mode='lines', name='Input Torque u2(t) (Nm)'))
    fig3.update_layout(
        title='Input Torques Applied to 2-DOF Robotic Arm',
        xaxis_title='Time (s)',
        yaxis_title='Torque (Nm)',
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig3.show()
